# Task 4 – Disease Prediction from Medical Data
### CodeAlpha Machine Learning Internship

**Objective:** Predict the possibility of **diabetes** in patients using classification algorithms.

**Dataset:** Pima Indians Diabetes Dataset (UCI ML Repository)

**Models:** Logistic Regression | Random Forest | SVM | XGBoost

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print('XGBoost available ✅')
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost not installed, skipping.')

print('All libraries imported ✅')

## 2. Load Dataset

In [ ]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
columns = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
           'Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']

try:
    df = pd.read_csv(url, header=None, names=columns)
    print('Dataset loaded from URL ✅')
except:
    np.random.seed(42)
    n = 768
    df = pd.DataFrame({
        'Pregnancies': np.random.randint(0,17,n),
        'Glucose': np.random.randint(50,200,n),
        'BloodPressure': np.random.randint(40,122,n),
        'SkinThickness': np.random.randint(5,100,n),
        'Insulin': np.random.randint(10,500,n),
        'BMI': np.round(np.random.uniform(18,67,n),1),
        'DiabetesPedigreeFunction': np.round(np.random.uniform(0.07,2.5,n),3),
        'Age': np.random.randint(21,81,n),
        'Outcome': np.random.randint(0,2,n)
    })
    print('Using synthetic demo data ⚠️')

print(f'Shape: {df.shape}')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('Dataset Info:')
df.info()
print('\nBasic Statistics:')
df.describe().round(2)

In [ ]:
print('Class Distribution:')
print(df['Outcome'].value_counts())
print(f'\nDiabetes rate: {df["Outcome"].mean():.2%}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Outcome'].value_counts().plot(kind='bar', ax=axes[0], color=['#4C72B0','#DD8452'], rot=0)
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['No Diabetes','Diabetes'])

corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1], square=True)
axes[1].set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Replace biologically impossible 0s with median
zero_cols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
df[zero_cols] = df[zero_cols].replace(0, np.nan)

print('Missing values after replacing 0s:')
print(df.isnull().sum())

df.fillna(df.median(numeric_only=True), inplace=True)
print('\nMissing values after imputation:')
print(df.isnull().sum().sum(), '← should be 0')

In [ ]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')

## 5. Train & Evaluate Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM':                 SVC(kernel='rbf', probability=True, random_state=42),
}
if XGBOOST_AVAILABLE:
    models['XGBoost'] = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')

results = {}
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    acc = accuracy_score(y_test, y_pred)
    cv  = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='accuracy').mean()
    results[name] = {'Accuracy': round(acc,4), 'CV Accuracy': round(cv,4)}
    print(f'\n{name}')
    print(f'  Test Accuracy: {acc:.4f}  |  CV Accuracy: {cv:.4f}')
    print(classification_report(y_test, y_pred, target_names=['No Diabetes','Diabetes']))

## 6. Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Disease Prediction – Model Analysis', fontsize=16, fontweight='bold')

# Model Accuracy Comparison
ax = axes[0,0]
names   = list(results.keys())
accs    = [results[n]['Accuracy'] for n in names]
cv_accs = [results[n]['CV Accuracy'] for n in names]
x = np.arange(len(names))
ax.bar(x-0.2, accs,    0.4, label='Test Accuracy', color='#4C72B0')
ax.bar(x+0.2, cv_accs, 0.4, label='CV Accuracy',   color='#DD8452')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylim(0,1); ax.set_title('Model Accuracy Comparison'); ax.legend()

# Confusion Matrix – best model
best_name  = max(results, key=lambda n: results[n]['Accuracy'])
y_pred_best = models[best_name].predict(X_test_sc)
cm = confusion_matrix(y_test, y_pred_best)
ConfusionMatrixDisplay(cm, display_labels=['No Diabetes','Diabetes']).plot(
    ax=axes[0,1], colorbar=False, cmap='Blues')
axes[0,1].set_title(f'Confusion Matrix – {best_name}')

# Feature Importance
feat_imp = pd.Series(
    models['Random Forest'].feature_importances_, index=X.columns
).sort_values(ascending=True)
feat_imp.plot(kind='barh', ax=axes[1,0], color='#55A868')
axes[1,0].set_title('Feature Importance (Random Forest)')

# Glucose vs BMI
sc = axes[1,1].scatter(df['Glucose'], df['BMI'], c=df['Outcome'],
                        cmap='coolwarm', alpha=0.5, edgecolors='none')
axes[1,1].set_xlabel('Glucose'); axes[1,1].set_ylabel('BMI')
axes[1,1].set_title('Glucose vs BMI  (Red = Diabetes)')
plt.colorbar(sc, ax=axes[1,1])

plt.tight_layout()
plt.savefig('disease_prediction_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved ✅')

## 7. Summary

In [ ]:
summary_df = pd.DataFrame(results).T
print('── Final Model Comparison ──')
print(summary_df.sort_values('Accuracy', ascending=False))
print(f'\n🏆 Best Model : {best_name}')
print(f'   Accuracy   : {results[best_name]["Accuracy"]:.4f}')